# Cargar_datos

**Proyecto final de Machine Learning: predicción de pago a tiempo de créditos**

En un entorno productivo, los datos vendrían del DWH o del Datalake de la empresa. Para este proyecto se usa un dataset no productivo de ejemplo en formato `.csv` (`Base_de_datos.csv`), ubicado en la raíz del repositorio.

Este notebook:
1. Define la ruta del archivo de forma que funcione desde cualquier carpeta.
2. Carga los datos con los tipos básicos correctos.
3. Hace una verificación inicial (dimensiones, tipos, nulos y duplicados).
4. Expone la función `cargar_datos()`, que reutilizan los demás componentes del pipeline.

## 1. Librerías

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)

## 2. Ruta del archivo

El notebook está dentro de `mlops_pipeline/` y el CSV está en la raíz del repositorio. Por eso se busca el archivo en la carpeta actual y en la carpeta superior, así funciona sin importar desde dónde se ejecute.

In [ ]:
NOMBRE_ARCHIVO = "Base_de_datos.csv"


def obtener_ruta_datos(nombre_archivo: str = NOMBRE_ARCHIVO) -> Path:
    """Busca el archivo de datos en la carpeta actual y en la carpeta superior."""
    for carpeta in [Path.cwd(), Path.cwd().parent]:
        ruta = carpeta / nombre_archivo
        if ruta.exists():
            return ruta
    raise FileNotFoundError(
        f"No se encontró '{nombre_archivo}'. Debe estar en la raíz del repositorio."
    )


RUTA_DATOS = obtener_ruta_datos()
print(f"Archivo encontrado en: {RUTA_DATOS}")

## 3. Función de carga

`fecha_prestamo` se lee directamente como fecha. La limpieza más profunda (unificar nulos, corregir valores inválidos, convertir categorías) se hace en `comprension_eda`.

In [ ]:
def cargar_datos(ruta: Path = None) -> pd.DataFrame:
    """Carga la base de créditos desde el CSV y devuelve un DataFrame."""
    if ruta is None:
        ruta = obtener_ruta_datos()
    df = pd.read_csv(ruta, parse_dates=["fecha_prestamo"], encoding="utf-8")
    return df


df = cargar_datos(RUTA_DATOS)
print(f"Datos cargados: {df.shape[0]:,} filas y {df.shape[1]} columnas")

## 4. Vista previa

In [ ]:
df.head()

In [ ]:
df.tail()

## 5. Tipos de datos y estructura

In [ ]:
df.info()

## 6. Verificaciones iniciales

In [ ]:
resumen = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "%_nulos": (df.isna().mean() * 100).round(2),
    "valores_unicos": df.nunique(),
})
resumen

In [ ]:
print(f"Filas duplicadas: {df.duplicated().sum()}")
print(f"Rango de fechas: {df['fecha_prestamo'].min()}  a  {df['fecha_prestamo'].max()}")

## 7. Variable objetivo

`Pago_atiempo`: **1** = el cliente pagó a tiempo, **0** = no pagó a tiempo.

In [ ]:
conteo = df["Pago_atiempo"].value_counts()
porcentaje = (df["Pago_atiempo"].value_counts(normalize=True) * 100).round(2)
pd.DataFrame({"cantidad": conteo, "porcentaje_%": porcentaje})

## 8. Observaciones para el EDA

Hallazgos de esta primera revisión, que se trabajan en `comprension_eda`:

- **Variable objetivo desbalanceada:** cerca del 95 % de los créditos se pagan a tiempo y solo cerca del 5 % no. Esto hay que tenerlo en cuenta al entrenar y evaluar (usar *recall*, *f1* y *roc_auc* en lugar de solo *accuracy*).
- **`tendencia_ingresos`:** debería tener solo `Creciente`, `Decreciente` y `Estable`, pero tiene algunos valores numéricos que son errores de captura. Hay que tratarlos como nulos.
- **`tipo_credito`:** es un código, así que es una variable **categórica** aunque aparezca como número. Tiene categorías con muy pocos registros (por ejemplo, el código 68 con un solo caso).
- **Nulos:** `promedio_ingresos_datacredito` y `tendencia_ingresos` tienen cerca del 27 % de nulos; `saldo_mora_codeudor`, `saldo_principal`, `saldo_mora`, `saldo_total` y `puntaje_datacredito` tienen pocos.
- **`fecha_prestamo`:** permite crear variables derivadas (mes, día de la semana, etc.).
- **`puntaje`:** hay que revisar en el EDA qué tan relacionado está con la variable objetivo, para descartar fuga de información (*data leakage*).